#Data Clean

In [1]:
!pip install lingua-language-detector
!pip install deep_translator
!pip install vaderSentiment

import pandas            as pd
import numpy             as np
import matplotlib.pyplot as plt
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

from deep_translator                 import GoogleTranslator
from lingua                          import Language, LanguageDetectorBuilder
from vaderSentiment.vaderSentiment   import SentimentIntensityAnalyzer
from collections import Counter

from google.colab import files, drive, userdata

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/ddds-cohort-21/Projects/Capstone/Data/chat_metrics_20260730_062950.xlsx'

dfs = pd.read_excel(file_path, sheet_name=None)
df = dfs['Raw Data']
feedback = dfs['User Feedback']

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.3/170.3 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.7 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
dfs = pd.read_excel(file_path, sheet_name=None)
df = dfs['Raw Data']
feedback = dfs['User Feedback']

In [3]:
#Raw Data
#Drop excess columns
df.drop(columns=['user_id', 'correlation_id', 'hour', 'date', 'secondary_category', 'step_summaries_count'], inplace=True)

#Convert from UTC to Mountain Time (Albuquerque)
df['timestamp'] = df['timestamp'].dt.tz_localize('UTC').dt.tz_convert('America/Denver')

#Update day of week to match local time
df['day_of_week'] = df['timestamp'].dt.day_name()

#Fill primary category nulls
df.loc[814, 'primary_category']= 'sunport_amenities'
df.loc[816, 'primary_category']= 'navigation'
df.loc[817, 'primary_category']= 'navigation'
df.loc[818, 'primary_category']= 'airline_logistics'

#Feedback
#Drop null star rating
feedback.drop([147, 262, 309, 311, 761], inplace=True)

#Drop unneccesary columns
feedback.drop(columns=['timestamp', 'ip_address', 'feedback_text'], inplace=True)

#Drop Duplicates
feedback = feedback.drop_duplicates()

#Merge
df = df.merge(feedback[['chat_id', 'star_rating']], on='chat_id', how='left')

#Emojis
emoji_pattern = re.compile(
    '['
    '\U0001F600-\U0001F64F'  # emoticons
    '\U0001F300-\U0001F5FF'  # symbols & pictographs
    '\U0001F680-\U0001F6FF'  # transport & map symbols
    '\U0001F1E0-\U0001F1FF'  # flags
    ']+',
    flags=re.UNICODE
)

#Find questions with emojis
emoji_questions = df[df['question'].astype(str).apply(lambda x: bool(emoji_pattern.search(x)))]

#Replace emojies with Okay
df['question'] = df['question'].replace('👍', 'Okay')

#Find different languages
detector = (LanguageDetectorBuilder.from_all_languages().build())

def detect_language(text, threshold=0.55):
  if not isinstance(text, str) or text.strip() == '':
    return 'ENGLISH'

  #Get confidence values for all languages
  confidence_values = detector.compute_language_confidence_values(text)

  #Check if highest matched meets your threshold
  if confidence_values and confidence_values[0].value > threshold:
    return confidence_values[0].language.name #Added name here

  return 'ENGLISH'

df['question_language'] = df['question'].apply(detect_language)
df['answer_language'] = df['answer'].apply(detect_language)

#Replace false positives
df['answer_language'] = df['answer_language'].replace(['LATIN', 'YORUBA', 'ESPERANTO'], 'ENGLISH')

#Translate
translator = GoogleTranslator(source='auto', target='en')

def translate_to_english(text):
  if not isinstance(text, str) or text.strip() == '':
    return text

  try:
    return translator.translate(text)
  except Exception:
    return text

#Translate questions
df['question_en'] = df['question']

mask = df['question_language'] != 'ENGLISH'

df.loc[mask, 'question_en'] = (df.loc[mask, 'question'].apply(translate_to_english))

#Translate answers
df['answer_en'] = df['answer']

mask = df['answer_language'] != 'ENGLISH'

df.loc[mask, 'answer_en'] = (df.loc[mask, 'answer'].apply(translate_to_english))

df['question'] = df['question_en']
df['answer'] = df['answer_en']

#Clean up and drop
df.drop(columns=['chat_id', 'question_language', 'answer_language', 'question_en', 'answer_en'], inplace=True)

#Save as Parquet

In [5]:
import os

In [9]:
parquet_file = 'df_clean.parquet'
df.to_parquet(parquet_file, index=False)
!ls -l --si {parquet_file}

os.environ["HF_TOKEN"] = userdata.get('hf_cs_token')
_ = os.environ["HF_TOKEN"]
f"{_[:5]} ... {_[-3:]}"

os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]

hf_org = "ddds-Capstone"
os.environ["HF_ORG"] = hf_org

hf_repo = "Datasets"
os.environ["HF_REPO"] = hf_repo

!hf auth login --token $HF_TOKEN

-rw-r--r-- 1 root root 3.0M Aug 20 01:17 df_clean.parquet
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `Capstone Token` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [10]:
%%capture hf_upload
%%bash
hf upload \
  --type dataset \
  ${HF_ORG}/${HF_REPO} \
  df_clean.parquet

In [11]:
hf_url = f"https://huggingface.co/datasets/{hf_org}/{hf_repo}/resolve/main/df_clean.parquet"
hf_url

'https://huggingface.co/datasets/ddds-Capstone/Datasets/resolve/main/df_clean.parquet'